In [2]:
import os
import re
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)


In [3]:
# File Paths
current_dir = os.getcwd()
data_path = f"{current_dir}/raw data"
ma_path = f"{data_path}/MA"
ny_path = f"{data_path}/NY"
sector_paths = ["FI", "HSA", "IN", "PST", "RT"]

In [62]:
# get link survival data from Linkedin
jp_links_filepath = f"{data_path}/jp_linkedin_links.csv"
jp_linkedin_df = pd.read_csv(jp_links_filepath)
links_df_l_status = jp_linkedin_df[jp_linkedin_df['status'].notna()]
links_df_dp = jp_linkedin_df[jp_linkedin_df['date_posted'].notna()]
# links_df_dp.to_csv(jp_links_filepath)
print(len(links_df_l_status))

379


In [11]:
# get link survival age data from Glassdoor + Indeed
jp_links_filepath = f"{data_path}/jp_links.csv"
links_df = pd.read_csv(jp_links_filepath)
links_df_dp = links_df[links_df['date_posted'].notna()]
links_df_status = links_df[links_df['status'].notna()]
print(len(links_df_status))
# links_df_other.to_csv(jp_links_filepath)

4672


In [12]:
links_df_no_status = links_df[links_df['status'].isna()]
links_df_no_status

,Unnamed: 0,id,site,job_url,job_url_direct,date_posted,status,last_checked_date,last_active_date,listing_age,listing_age_days,gd_salary
1218,2005,in-6938409785bc641f,indeed,https://www.indeed.com/viewjob?jk=6938409785bc641f,https://jsv3.recruitics.com/redirect?rx_cid=3436&rx_jobId=R-01328253_1003_rxr-3&rx_url=https%3A%...,2026-03-21,NaN,NaN,NaN,0,0.0,False
1588,3058,in-3e144d86c864f308,indeed,https://www.indeed.com/viewjob?jk=3e144d86c864f308,https://www.amazon.jobs/jobs/3169654/location-analyst-wwgs-growth--development?cmpid=DA_INAD200785B,2026-01-15,NaN,NaN,NaN,0,0.0,False
1589,3384,gd-1010056427301,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056427301,NaN,2026-03-06,NaN,NaN,NaN,0,0.0,False
1590,3385,gd-1010056202749,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056202749,NaN,2026-03-06,NaN,NaN,NaN,0,0.0,False
1591,3386,gd-1010055939472,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010055939472,NaN,2026-03-06,NaN,NaN,NaN,0,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...
24994,47556,in-f4fe5f63dc6efd84,indeed,https://www.indeed.com/viewjob?jk=f4fe5f63dc6efd84,https://jobs.dayforcehcm.com/en-US/visionworks/CANDIDATEPORTAL/jobs/241605,2026-03-09,NaN,NaN,NaN,0,0.0,False
24995,47557,in-84eca7039520b023,indeed,https://www.indeed.com/viewjob?jk=84eca7039520b023,https://jobs.dayforcehcm.com/en-US/visionworks/CANDIDATEPORTAL/jobs/241605,2026-03-09,NaN,NaN,NaN,0,0.0,False
24996,47558,in-7f2e5c42fb9917c5,indeed,https://www.indeed.com/viewjob?jk=7f2e5c42fb9917c5,https://jobs.dayforcehcm.com/en-US/visionworks/Retail/jobs/241606,2026-03-09,NaN,NaN,NaN,0,0.0,False
24997,47559,in-e6fb7b4af9648687,indeed,https://www.indeed.com/viewjob?jk=e6fb7b4af9648687,https://jobs.dayforcehcm.com/en-US/visionworks/Retail/jobs/241606,2026-03-09,NaN,NaN,NaN,0,0.0,False


In [5]:
# merge platform survival ages into one dataframe
all_links_df = pd.concat([links_df_status, links_df_l_status], ignore_index=True)
all_links_df

,Unnamed: 0,id,site,job_url,job_url_direct,date_posted,status,last_checked_date,last_active_date,listing_age,listing_age_days,gd_salary,Unnamed: 0.1,last_expired_date,relisted,reposted_date
0,0,gd-1010056379744,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056379744,NaN,2026-03-06,active,2026-03-24,NaN,0.0,0.0,False,NaN,NaN,NaN,NaN
1,1,gd-1010056098752,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056098752,NaN,2026-03-06,expired,2026-03-24,2026-03-24,0.0,18.0,False,NaN,NaN,NaN,NaN
2,2,gd-1010056244526,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056244526,NaN,2026-03-06,active,2026-03-24,NaN,0.0,0.0,False,NaN,NaN,NaN,NaN
3,3,gd-1010056045908,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056045908,NaN,2026-03-06,expired,2026-03-24,2026-03-24,0.0,18.0,False,NaN,NaN,NaN,NaN
4,4,gd-1010055442989,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010055442989,NaN,2026-03-05,active,2026-03-24,NaN,0.0,0.0,False,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2844,3,li-4389438416,linkedin,https://www.linkedin.com/jobs/view/4389438416,NaN,2026-03-24,active,03-25-2026,NaN,NaN,0.0,NaN,55557.0,NaN,False,NaN
2845,4,li-4389445330,linkedin,https://www.linkedin.com/jobs/view/4389445330,NaN,2026-03-24,active,03-25-2026,NaN,NaN,0.0,NaN,55558.0,NaN,False,NaN
2846,5,li-4389449386,linkedin,https://www.linkedin.com/jobs/view/4389449386,NaN,2026-03-24,active,03-25-2026,NaN,NaN,0.0,NaN,55559.0,NaN,False,NaN
2847,6,li-4379729877,linkedin,https://www.linkedin.com/jobs/view/4379729877,NaN,2026-03-23,active,03-25-2026,NaN,NaN,0.0,NaN,55560.0,NaN,True,NaN


In [20]:
# check for duplicates
duplicates = all_links_df[all_links_df.duplicated(subset=['id', 'job_url', 'site', 'date_posted', 'status'])]
all_links_df = all_links_df.drop_duplicates(subset=['id', 'job_url', 'site', 'date_posted', 'status'])
all_links_df = all_links_df.drop_duplicates(subset=['id'])
all_links_df

,Unnamed: 0,id,site,job_url,job_url_direct,date_posted,status,last_checked_date,last_active_date,listing_age,listing_age_days,gd_salary,Unnamed: 0.1,last_expired_date,relisted,reposted_date
0,0,gd-1010056379744,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056379744,NaN,2026-03-06,active,2026-03-24,NaN,0.0,0.0,False,NaN,NaN,NaN,NaN
1,1,gd-1010056098752,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056098752,NaN,2026-03-06,expired,2026-03-24,2026-03-24,0.0,18.0,False,NaN,NaN,NaN,NaN
2,2,gd-1010056244526,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056244526,NaN,2026-03-06,active,2026-03-24,NaN,0.0,0.0,False,NaN,NaN,NaN,NaN
3,3,gd-1010056045908,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056045908,NaN,2026-03-06,expired,2026-03-24,2026-03-24,0.0,18.0,False,NaN,NaN,NaN,NaN
4,4,gd-1010055442989,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010055442989,NaN,2026-03-05,active,2026-03-24,NaN,0.0,0.0,False,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2844,3,li-4389438416,linkedin,https://www.linkedin.com/jobs/view/4389438416,NaN,2026-03-24,active,03-25-2026,NaN,NaN,0.0,NaN,55557.0,NaN,False,NaN
2845,4,li-4389445330,linkedin,https://www.linkedin.com/jobs/view/4389445330,NaN,2026-03-24,active,03-25-2026,NaN,NaN,0.0,NaN,55558.0,NaN,False,NaN
2846,5,li-4389449386,linkedin,https://www.linkedin.com/jobs/view/4389449386,NaN,2026-03-24,active,03-25-2026,NaN,NaN,0.0,NaN,55559.0,NaN,False,NaN
2847,6,li-4379729877,linkedin,https://www.linkedin.com/jobs/view/4379729877,NaN,2026-03-23,active,03-25-2026,NaN,NaN,0.0,NaN,55560.0,NaN,True,NaN


In [13]:
# retrieve master dataset
master_dataset_filepath = f"{data_path}/master_dataset.csv"
master_dataset_df = pd.read_csv(master_dataset_filepath)
master_dataset_df = master_dataset_df.dropna(subset=['date_posted', 'description', 'location', 'company'])
master_dataset_df.head(5)

,Unnamed: 0,id,site,job_url,job_url_direct,title,company,location,date_posted,job_type,salary_source,interval,min_amount,max_amount,is_remote,job_level,job_function,listing_type,emails,description,company_industry,company_url,company_num_employees,company_revenue,company_description,sector,occupation,market,date_scraped
0,0,gd-1010056379744,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056379744,NaN,Associate Account Manager,The CCS Companies,"Norwood, MA",2026-03-06,NaN,direct_data,yearly,50000.0,57500.0,False,NaN,NaN,sponsored,NaN,"**Company Overview**\n\nThe CCS Companies is a leader in Business Process Outsourcing (BPO), pro...",NaN,https://www.glassdoor.com/Overview/W-EI_IE246357.htm,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06
1,1,gd-1010056098752,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056098752,NaN,Insurance Customer Service Representative,Express Employment Professionals,"Scituate, MA",2026-03-06,NaN,direct_data,hourly,25.0,30.0,False,NaN,NaN,sponsored,NaN,**Insurance business in Scituate is looking to grow their dedicated team.**\n\nIf you’d like to ...,NaN,NaN,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06
2,2,gd-1010056244526,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056244526,NaN,Senior Client Service Associate,TSW Wealth Management,"Framingham, MA",2026-03-06,NaN,direct_data,yearly,80000.0,100000.0,False,NaN,NaN,sponsored,NaN,TSW Wealth Management is a boutique wealth management firm based just outside of Boston in Frami...,NaN,NaN,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06
3,3,gd-1010056045908,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056045908,NaN,Customer Service Associate+,"Door Systems, Inc.","Framingham, MA",2026-03-06,NaN,direct_data,hourly,22.0,25.0,False,NaN,NaN,sponsored,NaN,"**Company Overview** \nDoor Systems, Inc. is a Framingham based, family owned and operated Gara...",NaN,https://www.glassdoor.com/Overview/W-EI_IE2584665.htm,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06
4,24,gd-1010055442989,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010055442989,NaN,Bilingual Multi-Location Customer Service Specialist (Spanish),Sherwin-Williams,"Pembroke, MA",2026-03-05,NaN,direct_data,hourly,21.0,21.0,False,NaN,NaN,sponsored,NaN,"*This position is eligible for health benefits, such as medical, dental and vision coverage, Fle...",NaN,https://www.glassdoor.com/Overview/W-EI_IE599.htm,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06


In [26]:
# check for duplicates
duplicates = master_dataset_df[master_dataset_df.duplicated(
    subset=['id', 'job_url', 'site', 'date_posted', 'job_url_direct', 'company', 
            'title', 'location'])]
master_dataset_df.drop_duplicates(
     subset=['id', 'job_url', 'site', 'date_posted', 'job_url_direct', 'company', 
            'title', 'location']
)
master_dataset_df = master_dataset_df.drop_duplicates(subset=master_dataset_df.columns.difference(['Unnamed: 0', 'sector', 'occupation', 'market', 'date_scraped']))
master_dataset_df

,Unnamed: 0,id,site,job_url,job_url_direct,title,company,location,date_posted,job_type,salary_source,interval,min_amount,max_amount,is_remote,job_level,job_function,listing_type,emails,description,company_industry,company_url,company_num_employees,company_revenue,company_description,sector,occupation,market,date_scraped
0,0,gd-1010056379744,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056379744,NaN,Associate Account Manager,The CCS Companies,"Norwood, MA",2026-03-06,NaN,direct_data,yearly,50000.0,57500.0,False,NaN,NaN,sponsored,NaN,"**Company Overview**\n\nThe CCS Companies is a leader in Business Process Outsourcing (BPO), pro...",NaN,https://www.glassdoor.com/Overview/W-EI_IE246357.htm,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06
1,1,gd-1010056098752,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056098752,NaN,Insurance Customer Service Representative,Express Employment Professionals,"Scituate, MA",2026-03-06,NaN,direct_data,hourly,25.0,30.0,False,NaN,NaN,sponsored,NaN,**Insurance business in Scituate is looking to grow their dedicated team.**\n\nIf you’d like to ...,NaN,NaN,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06
2,2,gd-1010056244526,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056244526,NaN,Senior Client Service Associate,TSW Wealth Management,"Framingham, MA",2026-03-06,NaN,direct_data,yearly,80000.0,100000.0,False,NaN,NaN,sponsored,NaN,TSW Wealth Management is a boutique wealth management firm based just outside of Boston in Frami...,NaN,NaN,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06
3,3,gd-1010056045908,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056045908,NaN,Customer Service Associate+,"Door Systems, Inc.","Framingham, MA",2026-03-06,NaN,direct_data,hourly,22.0,25.0,False,NaN,NaN,sponsored,NaN,"**Company Overview** \nDoor Systems, Inc. is a Framingham based, family owned and operated Gara...",NaN,https://www.glassdoor.com/Overview/W-EI_IE2584665.htm,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06
4,24,gd-1010055442989,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010055442989,NaN,Bilingual Multi-Location Customer Service Specialist (Spanish),Sherwin-Williams,"Pembroke, MA",2026-03-05,NaN,direct_data,hourly,21.0,21.0,False,NaN,NaN,sponsored,NaN,"*This position is eligible for health benefits, such as medical, dental and vision coverage, Fle...",NaN,https://www.glassdoor.com/Overview/W-EI_IE599.htm,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124577,3,li-4389438416,linkedin,https://www.linkedin.com/jobs/view/4389438416,NaN,"Store Associate (Part Time) - Garden City Park, NY",Lidl US,"New Hyde Park, NY",2026-03-24,parttime,NaN,NaN,NaN,NaN,False,entry level,Sales and Business Development,NaN,NaN,**Summary**\n Store Associates provide our customers with the shopping experience that Lidl is f...,Retail,https://www.linkedin.com/company/lidl-us,NaN,NaN,NaN,RT,STO,NY,2026-03-25
124578,4,li-4389445330,linkedin,https://www.linkedin.com/jobs/view/4389445330,NaN,Part Time Stock Coordinator (Greenwich),Buck Mason,"Greenwich, CT",2026-03-24,parttime,NaN,NaN,NaN,NaN,False,not applicable,Other,NaN,NaN,**Keep Buck Mason Running Smooth as a Stock Coordinator!**\n Are you the behind\\-the\\-scenes h...,Retail Apparel and Fashion,https://www.linkedin.com/company/buck-mason,NaN,NaN,NaN,RT,STO,NY,2026-03-25
124579,5,li-4389449386,linkedin,https://www.linkedin.com/jobs/view/4389449386,NaN,Retail Stocking Associate - Part Time,"Burlington Stores, Inc.","Massapequa, NY",2026-03-24,parttime,NaN,NaN,NaN,NaN,False,entry level,Management and Manufacturing,NaN,NaN,"If you want an exciting job with one of the largest off\\-price retail stores in the nation, joi...",Retail,https://www.linkedin.com/company/burlington-stores,NaN,NaN,NaN,RT,STO,NY,2026-03-25
124580,6,li-4379729877,linkedin,https://www.linkedin.com/jobs/view/4379729877,NaN,"Overnight Grocery Team Member (Stocker, Inventory, Receiving) - Part Time",Whole Fo

In [41]:
# merge master dataset with listing age data
merged_df = pd.merge(master_dataset_df, all_links_df, on="id", how="right")
cols_to_drop = ['company', 'company_industry', 'company_num_employees', 'company_revenue', 'company_description', 'location', 'date_posted_y', 'site_y', 'salary_source', 'job_url_direct_y', 'job_url_y', 'gd_salary', 'company_url', 'emails', 'job_function', 'job_level', 'salar_source', 'job_type', 'Unnamed:0_x', 'Unnamed: 0_y' 'Unnamed: 0.1']
merged_df = merged_df.drop(columns=cols_to_drop, axis=1, errors='ignore')
print(len(merged_df))
merged_df.head(5)

2815


,Unnamed: 0_x,id,site_x,job_url_x,job_url_direct_x,title,date_posted_x,interval,min_amount,max_amount,is_remote,listing_type,description,sector,occupation,market,date_scraped,Unnamed: 0_y,status,last_checked_date,last_active_date,listing_age,listing_age_days,Unnamed: 0.1,last_expired_date,relisted,reposted_date
0,0,gd-1010056379744,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056379744,NaN,Associate Account Manager,2026-03-06,yearly,50000.0,57500.0,False,sponsored,"**Company Overview**\n\nThe CCS Companies is a leader in Business Process Outsourcing (BPO), pro...",FI,CSR-FI,MA,2026-03-06,0,active,2026-03-24,NaN,0.0,0.0,NaN,NaN,NaN,NaN
1,1,gd-1010056098752,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056098752,NaN,Insurance Customer Service Representative,2026-03-06,hourly,25.0,30.0,False,sponsored,**Insurance business in Scituate is looking to grow their dedicated team.**\n\nIf you’d like to ...,FI,CSR-FI,MA,2026-03-06,1,expired,2026-03-24,2026-03-24,0.0,18.0,NaN,NaN,NaN,NaN
2,2,gd-1010056244526,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056244526,NaN,Senior Client Service Associate,2026-03-06,yearly,80000.0,100000.0,False,sponsored,TSW Wealth Management is a boutique wealth management firm based just outside of Boston in Frami...,FI,CSR-FI,MA,2026-03-06,2,active,2026-03-24,NaN,0.0,0.0,NaN,NaN,NaN,NaN
3,3,gd-1010056045908,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056045908,NaN,Customer Service Associate+,2026-03-06,hourly,22.0,25.0,False,sponsored,"**Company Overview** \nDoor Systems, Inc. is a Framingham based, family owned and operated Gara...",FI,CSR-FI,MA,2026-03-06,3,expired,2026-03-24,2026-03-24,0.0,18.0,NaN,NaN,NaN,NaN
4,24,gd-1010055442989,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010055442989,NaN,Bilingual Multi-Location Customer Service Specialist (Spanish),2026-03-05,hourly,21.0,21.0,False,sponsored,"*This position is eligible for health benefits, such as medical, dental and vision coverage, Fle...",FI,CSR-FI,MA,2026-03-06,4,active,2026-03-24,NaN,0.0,0.0,NaN,NaN,NaN,NaN


In [61]:
# data statistics
merged_df_fixed = merged_df.copy()

merged_b_df = merged_df[merged_df['status'] == "blocked"]
merged_rm_df = merged_df[merged_df['status'] == "removed"]
merged_exp_df = merged_df[merged_df['status'] == "expired"]
merged_act_df = merged_df[merged_df['status'] == "active"]
print(f"Blocked: {len(merged_b_df)}, Removed: {len(merged_b_df)}, Expired: {len(merged_exp_df)}, Active: {len(merged_act_df)}")

merged_df_fixed = merged_df_fixed[merged_df_fixed['status'] != "blocked"]
merged_df_fixed = merged_df.reset_index()

Blocked: 16, Removed: 16, Expired: 1064, Active: 1716


In [54]:
# investigate observations that were blocked and update values - to do
print(len(merged_b_df))
merged_b_df.head(1)

16


,Unnamed: 0_x,id,site_x,job_url_x,job_url_direct_x,title,date_posted_x,interval,min_amount,max_amount,is_remote,listing_type,description,sector,occupation,market,date_scraped,Unnamed: 0_y,status,last_checked_date,last_active_date,listing_age,listing_age_days,Unnamed: 0.1,last_expired_date,relisted,reposted_date
136,23,gd-1010063001457,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010063001457,NaN,Client Engagement Coordinator,2026-03-12,yearly,80000.0,100000.0,False,organic,"Defense Holdings, Inc. (DHi) \n\nLocation: Hybrid (US) \n\nEmployment Type: Full\\-Time \n\nD...",FI,CSR-FI,MA,2026-03-12,133,blocked,2026-03-24,NaN,0.0,0.0,NaN,NaN,NaN,NaN


In [35]:
merged_rm_df = merged_df[merged_df['status'] == "removed"]
print(len(merged_rm_df))
merged_rm_df

18


,Unnamed: 0_x,id,site_x,job_url_x,job_url_direct_x,title,company,location,date_posted_x,interval,min_amount,max_amount,is_remote,listing_type,description,company_industry,company_num_employees,company_revenue,company_description,sector,occupation,market,date_scraped,Unnamed: 0_y,status,last_checked_date,last_active_date,listing_age,listing_age_days,Unnamed: 0.1,last_expired_date,relisted,reposted_date
32,1,gd-1010058270833,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010058270833,NaN,"Manager 2, QuickBooks Live",Intuit,"Boston, MA",2026-03-07,NaN,NaN,NaN,False,organic,**Overview**\n------------\n\n\nJoin the Intuit Customer Success team as a Senior Manager of Cus...,NaN,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-07,31,removed,2026-03-24,NaN,0.0,0.0,NaN,NaN,NaN,NaN
174,8,gd-1010066310680,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010066310680,NaN,Regional Accounts Coordinator,Big Grove Brewery,United States,2026-03-15,yearly,50917.0,61320.0,False,sponsored,**Job Overview**\n\nBig Grove Brewery is seeking a dynamic and highly organized Regional Account...,NaN,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-16,171,removed,2026-03-24,NaN,0.0,0.0,NaN,NaN,NaN,NaN
273,2,in-72ebd5876874706e,indeed,https://www.indeed.com/viewjob?jk=72ebd5876874706e,https://dsp.prng.co/yKMI26b,"Manager 2, QuickBooks Live",Intuit,"Boston, MA, US",2026-03-07,NaN,NaN,NaN,True,NaN,**Overview**\n------------\n\n\nJoin the Intuit Customer Success team as a Senior Manager of Cus...,NaN,"10,000+",more than $10B (USD),Intuit is the global financial technology platform that powers prosperity for the people and com...,FI,CSR-FI,MA,2026-03-07,269,removed,2026-03-25,NaN,0.0,0.0,NaN,NaN,NaN,NaN
344,11,in-2ca569224e87c733,indeed,https://www.indeed.com/viewjob?jk=2ca569224e87c733,https://click.appcast.io/t/Su8rE_GriodrEqRAeCRQCQlTL5lemlpci0ZjT_1GnTQ=,RETAIL CUSTOMER SERVICE LEADER (FRONT END LEADER),Micro Center,"Cambridge, MA, US",2026-03-09,hourly,18.0,23.0,False,NaN,**MICRO CENTER** is the nation’s leading computer and electronic device big box retailer! Our te...,NaN,"1,001 to 5,000",$1B to $5B (USD),Micro Center sells computers and consumer electronics through 26 stores in 20 states.,FI,CSR-FI,MA,2026-03-10,347,removed,2026-03-25,NaN,0.0,0.0,NaN,NaN,NaN,NaN
446,61,in-bea93bae2005a6ba,indeed,https://www.indeed.com/viewjob?jk=bea93bae2005a6ba,https://ferguson.wd1.myworkdayjobs.com/Ferguson_Experienced/job/Franklin-MA/Showroom-Customer-Se...,Showroom Customer Service Representative,Ferguson,"Franklin, MA, US",2026-03-10,hourly,22.0,36.0,False,NaN,"**Job Description:**\n\nSince 1953, Ferguson has been a source of quality supplies for a variety...",NaN,"10,000+",Decline to state,Ferguson is the largest value-added distributor serving the water and air specialized profession...,FI,CSR-FI,MA,2026-03-11,449,removed,2026-03-25,NaN,0.0,0.0,NaN,NaN,NaN,NaN
447,62,in-58b1806804d7e05e,indeed,https://www.indeed.com/viewjob?jk=58b1806804d7e05e,https://ferguson.wd1.myworkdayjobs.com/Ferguson_Experienced/job/Newton-MA/Showroom-Customer-Serv...,Showroom Customer Service Representative,Ferguson,"Newton, MA, US",2026-03-10,hourly,23.0,38.0,False,NaN,"**Job Description:**\n\nSince 1953, Ferguson has been a source of quality supplies for a variety...",NaN,"10,000+",Decline to state,Ferguson is the largest value-added distributor serving the water and air specialized profession...,FI,CSR-FI,MA,2026-03-11,450,removed,2026-03-25,NaN,0.0,0.0,NaN,NaN,NaN,NaN
745,62,in-3321bfd1cfb929e1,indeed,https://www.indeed.com/viewjob?jk=3321bfd1cfb929e1,https://workwithus.circlek.com/global/en/job/CIKCGLOBALR572016EXTERNALENGLOBAL/Customer-Service-...,Customer Service Representative,Circle K,"Derry, NH, US",2026-03-17,NaN,NaN,NaN,False,NaN,"Store 4707234: 55 Bypass 28, Derry, New Hampshire 03038**Shift Availability**\n=================...",NaN,"10,000+",more than $10B (USD),"Circle K/Couche-Tard is a global leader in convenience and mobility, operating in 29 countries a...",FI,CSR-FI,MA,2026-03-18,746,remov

In [36]:
merged_exp_df = merged_df[merged_df['status'] == "expired"]
print(len(merged_exp_df))
merged_exp_df

1064


,Unnamed: 0_x,id,site_x,job_url_x,job_url_direct_x,title,company,location,date_posted_x,interval,min_amount,max_amount,is_remote,listing_type,description,company_industry,company_num_employees,company_revenue,company_description,sector,occupation,market,date_scraped,Unnamed: 0_y,status,last_checked_date,last_active_date,listing_age,listing_age_days,Unnamed: 0.1,last_expired_date,relisted,reposted_date
1,1,gd-1010056098752,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056098752,NaN,Insurance Customer Service Representative,Express Employment Professionals,"Scituate, MA",2026-03-06,hourly,25.0,30.0,False,sponsored,**Insurance business in Scituate is looking to grow their dedicated team.**\n\nIf you’d like to ...,NaN,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06,1,expired,2026-03-24,2026-03-24,0.0,18.0,NaN,NaN,NaN,NaN
3,3,gd-1010056045908,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056045908,NaN,Customer Service Associate+,"Door Systems, Inc.","Framingham, MA",2026-03-06,hourly,22.0,25.0,False,sponsored,"**Company Overview** \nDoor Systems, Inc. is a Framingham based, family owned and operated Gara...",NaN,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06,3,expired,2026-03-24,2026-03-24,0.0,18.0,NaN,NaN,NaN,NaN
6,26,gd-1010055443368,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010055443368,NaN,Customer Service Associate - Automotive Finishes,Sherwin-Williams,"Randolph, MA",2026-03-05,hourly,19.0,19.0,False,sponsored,The Customer Service Associate is responsible for delivering products to customers from Sherwin\...,NaN,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06,6,expired,2026-03-24,2026-03-24,0.0,19.0,NaN,NaN,NaN,NaN
7,38,gd-1010053110102,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010053110102,NaN,Client Service Representative,LOCATEPLUS,"Danvers, MA",2026-03-04,yearly,50000.0,55000.0,False,sponsored,LocatePLUS is the nation's leading provider of investigative databases. We are looking for a tal...,NaN,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06,7,expired,2026-03-24,2026-03-24,0.0,20.0,NaN,NaN,NaN,NaN
8,39,gd-1010052851185,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010052851185,NaN,Customer Service Representative,W.T. Hight Company,"Pembroke, MA",2026-03-04,hourly,18.0,21.0,False,sponsored,"**Job Summary** \nWe're looking for a friendly, organized, and proactive Customer Service Repre...",NaN,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06,8,expired,2026-03-24,2026-03-24,0.0,20.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2789,0,li-4378652620,linkedin,https://www.linkedin.com/jobs/view/4378652620,NaN,Retail Stocking Associate,Harbor Freight Tools,"Brooklyn, NY",2026-03-21,NaN,NaN,NaN,False,NaN,A Retail Stocking Associate (part\\-time) is a valued member of a high performing team who is em...,Retail,NaN,NaN,NaN,RT,STO,NY,2026-03-22,0,expired,03-25-2026,NaN,NaN,0.0,55536.0,NaN,True,03-21-2026
2796,7,li-4388298301,linkedin,https://www.linkedin.com/jobs/view/4388298301,NaN,Dairy Stocker,Wegmans Food Markets,"Manalapan, NJ",2026-03-21,NaN,NaN,NaN,False,NaN,"Our mission is to provide incredible service and help our customers live healthier, better lives...",Retail,NaN,NaN,NaN,RT,STO,NY,2026-03-22,7,expired,03-25-2026,NaN,NaN,0.0,55543.0,NaN,False,NaN
2800,11,li-4387997593,linkedin,https://www.linkedin.com/jobs/view/4387997593,NaN,Retail Stocking Associate - Part Time,"Burlington Stores, Inc.","Vauxhall, NJ",2026-03-20,NaN,NaN,NaN,False,NaN,"If you want an exciting job with one of the largest off\\-price retail stores in the nation, joi...",Retail,NaN,NaN,NaN,RT,STO,NY,2026-03-22,11,expired,03-25-2026,NaN,NaN,0.0,55547.0,NaN,False,NaN
2806,3,li-4389424082,linkedin,https://www.linkedin.com/jobs/view/4389424082,NaN,Seasonal Receiver Stocker Days,"Lowe's Companies, Inc.","Jersey City, NJ",2026-03-24,NaN,NaN,NaN,False,NaN,**Key Responsibilities**\n* Provides SMART customer service at all times through the daily execu...,Retail,NaN

In [37]:
merged_act_df = merged_df[merged_df['status'] == "active"]
print(len(merged_act_df))
merged_act_df

1716


,Unnamed: 0_x,id,site_x,job_url_x,job_url_direct_x,title,company,location,date_posted_x,interval,min_amount,max_amount,is_remote,listing_type,description,company_industry,company_num_employees,company_revenue,company_description,sector,occupation,market,date_scraped,Unnamed: 0_y,status,last_checked_date,last_active_date,listing_age,listing_age_days,Unnamed: 0.1,last_expired_date,relisted,reposted_date
0,0,gd-1010056379744,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056379744,NaN,Associate Account Manager,The CCS Companies,"Norwood, MA",2026-03-06,yearly,50000.0,57500.0,False,sponsored,"**Company Overview**\n\nThe CCS Companies is a leader in Business Process Outsourcing (BPO), pro...",NaN,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06,0,active,2026-03-24,NaN,0.0,0.0,NaN,NaN,NaN,NaN
2,2,gd-1010056244526,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056244526,NaN,Senior Client Service Associate,TSW Wealth Management,"Framingham, MA",2026-03-06,yearly,80000.0,100000.0,False,sponsored,TSW Wealth Management is a boutique wealth management firm based just outside of Boston in Frami...,NaN,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06,2,active,2026-03-24,NaN,0.0,0.0,NaN,NaN,NaN,NaN
4,24,gd-1010055442989,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010055442989,NaN,Bilingual Multi-Location Customer Service Specialist (Spanish),Sherwin-Williams,"Pembroke, MA",2026-03-05,hourly,21.0,21.0,False,sponsored,"*This position is eligible for health benefits, such as medical, dental and vision coverage, Fle...",NaN,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06,4,active,2026-03-24,NaN,0.0,0.0,NaN,NaN,NaN,NaN
5,25,gd-1010055442978,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010055442978,NaN,Bilingual Multi-Location Customer Service Specialist (Spanish),Sherwin-Williams,"Lynn, MA",2026-03-05,hourly,21.0,21.0,False,sponsored,"*This position is eligible for health benefits, such as medical, dental and vision coverage, Fle...",NaN,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06,5,active,2026-03-24,NaN,0.0,0.0,NaN,NaN,NaN,NaN
10,41,gd-1010053063299,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010053063299,NaN,Customer Service Specialist,Reynolds and Reynolds,"North Andover, MA",2026-03-04,yearly,50000.0,65000.0,False,sponsored,**\\*In office position working Monday\\-Friday\\***\n\n**Shift Options: 4 days 8am\\-5pm and 1 ...,NaN,NaN,NaN,NaN,FI,CSR-FI,MA,2026-03-06,10,active,2026-03-24,NaN,0.0,0.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2810,4,li-4389445330,linkedin,https://www.linkedin.com/jobs/view/4389445330,NaN,Part Time Stock Coordinator (Greenwich),Buck Mason,"Greenwich, CT",2026-03-24,NaN,NaN,NaN,False,NaN,**Keep Buck Mason Running Smooth as a Stock Coordinator!**\n Are you the behind\\-the\\-scenes h...,Retail Apparel and Fashion,NaN,NaN,NaN,RT,STO,NY,2026-03-25,4,active,03-25-2026,NaN,NaN,0.0,55558.0,NaN,False,NaN
2811,5,li-4389449386,linkedin,https://www.linkedin.com/jobs/view/4389449386,NaN,Retail Stocking Associate - Part Time,"Burlington Stores, Inc.","Massapequa, NY",2026-03-24,NaN,NaN,NaN,False,NaN,"If you want an exciting job with one of the largest off\\-price retail stores in the nation, joi...",Retail,NaN,NaN,NaN,RT,STO,NY,2026-03-25,5,active,03-25-2026,NaN,NaN,0.0,55559.0,NaN,False,NaN
2812,15,li-4379729877,linkedin,https://www.linkedin.com/jobs/view/4379729877,NaN,"Overnight Grocery Team Member (Stocker, Inventory, Receiving) - Part Time",Whole Foods Market,"New York, NY",2026-03-02,NaN,NaN,NaN,False,NaN,Provides overnight support for assigned team to include receiving and preparing product and main...,Retail,NaN,NaN,NaN,RT,STO,NY,2026-03-05,6,active,03-25-2026,NaN,NaN,0.0,55560.0,NaN,True,NaN
2813,6,li-4379729877,linkedin,https://www.linkedin.com/jobs/view/4379729877,NaN,"Overnight Grocery Team Member (Stocker, Inventory, Receiving) - Part Time",Whole Foods Market,"New York, NY",2026-03-23,NaN,NaN,NaN,False,NaN,

In [57]:
working_df = pd.DataFrame({
    'id': pd.Series(dtype='str'),
    'glassdoor':  pd.Series(dtype='int32'),
    'linkedin':  pd.Series(dtype='int32'),
    'indeed':  pd.Series(dtype='int32'),
    'ny':  pd.Series(dtype='int32'),
    'ma':  pd.Series(dtype='int32'),
    'remote':  pd.Series(dtype='int32'),
    'listing_type':  pd.Series(dtype='int32'),
    'fi':  pd.Series(dtype='int32'),
    'rt':  pd.Series(dtype='int32'),
    'in':  pd.Series(dtype='int32'),
    'hsa':  pd.Series(dtype='int32'),
    'pst':  pd.Series(dtype='int32'),
    'relisted':  pd.Series(dtype='int32'),
    'listing_age':  pd.Series(dtype='int32'),
    'listing_age_days':  pd.Series(dtype='int32'),
    'salary_range':  pd.Series(dtype='int32'),
    'salary_base':  pd.Series(dtype='int32'), 
    'no_salary':  pd.Series(dtype='int32'),
    'salary_min':  pd.Series(dtype='int32'),
    'salary_max':  pd.Series(dtype='int32'),
    'salary_base':  pd.Series(dtype='int32'),
})

In [ ]:
def standardize_vals(var_column):
    norm_col = np.array()
    for i in range(len(var_column)):
        if i is not None:
            np.append(norm_col, [1])
        else:
            np.append(norm_col, [0])
    return pd.Series(norm_col)
 


Unnamed: 0_x
